In [ ]:
!pip install -q sentence-transformers chromadb google-genai pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
import fitz
from google import genai
from google.colab import files, userdata

In [ ]:
client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

In [ ]:
def extract_text(pdf_path):
    text = ""

    pdf = fitz.open(pdf_path)

    for page in pdf:
        text += page.get_text()

    pdf.close()

    return text

In [ ]:
def create_chunks(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])

    return chunks

In [ ]:
uploaded = files.upload()

Saving JEEVANRAJ S S- RESUME.pdf to JEEVANRAJ S S- RESUME.pdf
Saving gayathriresume1 (2).pdf to gayathriresume1 (2).pdf
Saving Dhanuja Resume.pdf to Dhanuja Resume.pdf
Saving Anshif_Resume.pdf to Anshif_Resume.pdf


In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(
    name="resume_collection"
)

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
doc_id = 0

for filename in uploaded.keys():

    text = extract_text(filename)

    chunks = create_chunks(text)

    for chunk in chunks:

        embedding = model.encode(chunk).tolist()

        collection.add(
            ids=[str(doc_id)],
            embeddings=[embedding],
            documents=[chunk],
            metadatas=[
                {
                    "candidate": filename
                }
            ]
        )

        doc_id += 1

In [ ]:
job_description = input(
    "Enter Job Description: "
)

Enter Job Description: AI Engineer


In [ ]:
query = f"""
Compare all candidates for this role:

{job_description}

Rank candidates from best to worst.
"""

In [ ]:
query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=15
)

In [ ]:
context = ""

documents = results["documents"][0]
metadatas = results["metadatas"][0]

for doc, meta in zip(documents, metadatas):

    context += (
        f"\nCandidate: {meta['candidate']}\n"
        f"{doc}\n"
    )

In [ ]:
prompt = f"""
You are a senior HR recruiter.

Analyze all candidates based on the retrieved resume information.

Tasks:

1. Compare all candidates.
2. Rank them from best to worst.
3. Give suitability scores out of 100.
4. Mention strengths.
5. Mention weaknesses.
6. Recommend the best candidate.
7. Generate a comparison table.

Job Description:
{job_description}

Resume Context:
{context}
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config={
        "temperature":0.2
    }
)

print(response.text)

As a senior HR recruiter, I have analyzed the provided resume snippets for the AI Engineer position. The role requires strong technical skills in Python, LLMs, NLP, Machine Learning, Flask/FastAPI, and Cloud Services, along with practical application and problem-solving abilities.

Here's a detailed analysis, comparison, ranking, and recommendation for the candidates:

---

### **1. Candidate Comparison & Analysis**

**Dhanuja Vijay Anand**
*   **Education:** B.Tech Artificial Intelligence and Data Science, Karpagam College of Engineering.
*   **Summary:** Strong foundation in Python, ML, data analysis, generative AI. Passionate about applying AI to real-world problems. Seeking internship.
*   **Key Skills:** Python, Data Structures & Algorithms, Machine Learning fundamentals, TensorFlow, Keras, OpenCV, CNN, LLMs, NLP, Flask/FastAPI, Cloud Services. Problem-solving, communication.
*   **Key Projects/Experience:**
    *   **MindMate AI:** Generative AI Mental Health Companion (Python, L